# StageOps: limpieza de `customers.csv`

**Hackatón SO-2026-041 · Investigación de incidencias en la venta de entradas**

`customers.csv` es el catálogo de **compradores**: una fila por comprador, con `customer_id` (clave primaria), nombre, apellidos y correo ficticio en `example.com`. No incluye a todos los acompañantes, solo a quienes realizaron la compra. El mismo comprador puede aparecer en varias reservas (incluso de distintos conciertos), por lo que `customer_id` es la clave que une compras, dinero y accesos potenciales.

`CSV original → Inspección → Tipado → Validación (con sistema de comprobantes) → Detección de homónimos → Tabla limpia → Exportación`

El CSV original (`data/raw/customers.csv`) no se modifica. El resultado limpio se guarda en `data/clean/customers_clean.csv`.

## 1. Método de trabajo

1. **Inspeccionar** el fichero tal como llega (filas, columnas, tipos, valores ausentes).
2. **Tipar** todas las columnas como texto (estándar del catálogo de compradores).
3. **Validar con sistema de comprobantes** (`comprobante` avanza solo si cada paso pasa):
   - `customer_id`: vacío, formato `CLI-<números>`, longitud de sufijo consistente, duplicados (clave primaria), secuencia incremental y gaps.
   - `first_name` y `last_name`: vacío, caracteres inusuales.
   - `email`: vacío, formato, dominio `@example.com`, duplicados.
4. **Detectar homónimos** (mismo nombre+apellido, distinto `customer_id`): advertencia, no error, porque el anexo permite homónimos.
5. **Construir tabla de calidad** (completitud y unicidad) del dataset original.
6. **Limpiar y exportar** (`limpiar_customers.py`), guardando `customers_clean.csv` sin alterar filas ni columnas si no hay nada que corregir.
7. **Validar con `assert`** y **releer** para comprobar que el fichero exportado conserva filas y columnas.

Toda la ejecución es reproducible por código; no depende de revisión visual.

## 2. Importaciones y ubicación de los archivos

La raíz del repositorio se localiza subiendo desde la carpeta de trabajo hasta encontrar `data/raw`. Las rutas se definen una sola vez como constantes.

In [ ]:
from pathlib import Path
import pandas as pd
import re

CARPETA_BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').is_dir())
RUTA_ORIGINAL = CARPETA_BASE / 'data' / 'raw' / 'customers.csv'
CARPETA_LIMPIOS = CARPETA_BASE / 'data' / 'clean'
RUTA_LIMPIO = CARPETA_LIMPIOS / 'customers_clean.csv'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

if not RUTA_ORIGINAL.exists():
    raise FileNotFoundError(f'No se encuentra {RUTA_ORIGINAL}. Revisa el directorio de trabajo: {Path.cwd()}')
CARPETA_LIMPIOS.mkdir(parents=True, exist_ok=True)
print('Carpeta base:', CARPETA_BASE)


## 3. Inspección del fichero original

Se lee con `encoding='utf-8-sig'` porque el CSV lleva BOM. Todas las columnas se leen como texto (`str`) para no perder ceros a la izquierda ni forzar conversiones.

In [ ]:
df_raw = pd.read_csv(
    RUTA_ORIGINAL,
    dtype={
        'customer_id': str,
        'first_name': str,
        'last_name': str,
        'email': str,
    },
    encoding='utf-8-sig',
    keep_default_na=False,
    na_values=[''],
)

print(f'Registros: {len(df_raw):,} | columnas: {list(df_raw.columns)}\n')
df_raw.info()
display(df_raw.head(10))
print('Valores ausentes por columna:', df_raw.isna().sum()[lambda s: s > 0].to_dict() or 'ninguno')


**Lo que muestran los datos:** 59.932 compradores, cuatro columnas y ningún valor ausente. `customer_id` tiene 59.932 valores únicos (clave primaria válida). `first_name` tiene solo 40 valores únicos (hay muchos homónimos), `last_name` tiene 1.296, y `email` tiene 59.932 únicos (sin duplicados). Todos los correos usan `example.com`.

## 4. Tipado y normalización

Se trabaja sobre una copia para conservar `df_raw` intacto. Todas las columnas ya son texto; se eliminan espacios sobrantes y se fuerza el tipo `string` de pandas para garantizar consistencia.

In [ ]:
df = df_raw.copy()

for columna in df.columns:
    df[columna] = df[columna].astype('string').str.strip()

df.dtypes.to_frame('tipo')


## 5. Validaciones de calidad (sistema de comprobantes)

Cada comprobación cuenta cuántas filas la **incumplen**. Se usa una variable `comprobante` que avanza solo si cada paso pasa; si falla, se detiene en 0. Las columnas de conteo se almacenan para el resumen final.

In [ ]:
errores = []
advertencias = []

def registrar(tipo, mensaje):
    if tipo == 'error':
        errores.append(mensaje)
        print(f'[ERROR] {mensaje}')
    else:
        advertencias.append(mensaje)
        print(f'[AVISO] {mensaje}')

comprobaciones = {}

print('--- Validación: customer_id ---')
cid = df['customer_id'].astype(str).str.strip()

# Comprobante 1: sin celdas vacías
n_vacios = int((cid == '').sum())
comprobaciones['customer_id_vacío'] = n_vacios
if n_vacios > 0:
    registrar('error', f'customer_id vacío: {n_vacios} filas')
else:
    print('  customer_id sin celdas vacías: OK')
    comprobante = 1

# Comprobante 2: formato CLI-<números>
if comprobante == 1:
    patron = cid.str.extract(r'^CLI-(\d+)$')[0]
    valido_formato = patron.notna()
    n_formato_invalido = int((~valido_formato).sum())
    comprobaciones['customer_id_formato_inválido'] = n_formato_invalido
    if n_formato_invalido > 0:
        ejemplos = df.loc[~valido_formato, 'customer_id'].head(10).tolist()
        registrar('error', f'customer_id con formato inválido (no es CLI-<números>): {n_formato_invalido} filas. Ejemplos: {ejemplos}')
    else:
        print('  customer_id con formato CLI-<números>: OK')
        comprobante = 2

# Comprobante 3: longitud de sufijo consistente
if comprobante == 2:
    longitudes = patron[valido_formato].str.len()
    long_unica = longitudes.unique()
    if len(long_unica) > 1:
        contador_long = longitudes.value_counts().to_dict()
        registrar('error', f'customer_id con longitudes de sufijo inconsistentes: {contador_long}')
    else:
        print(f'  Longitud de sufijo numérico consistente: {long_unica[0]} dígitos: OK')
        comprobante = 3

# Comprobante 4: sin duplicados (clave primaria)
if comprobante == 3:
    n_dup = int(cid.duplicated(keep=False).sum())
    comprobaciones['customer_id_duplicado'] = n_dup
    if n_dup > 0:
        registrar('error', f'customer_id duplicado: {n_dup} filas')
    else:
        print('  customer_id sin duplicados (clave primaria): OK')
        comprobante = 4

# Comprobante 5: secuencia incremental y gaps
if comprobante == 4:
    numeros = patron.astype(int)
    min_n = int(numeros.min())
    max_n = int(numeros.max())
    esperado = set(range(min_n, max_n + 1))
    presentes = set(numeros.tolist())
    faltantes = sorted(esperado - presentes)
    comprobaciones['customer_id_gaps'] = len(faltantes)
    print(f'  Rango incremental: {min_n} a {max_n} (esperados: {max_n - min_n + 1:,}, presentes: {len(presentes):,})')
    if faltantes:
        registrar('error', f'Gaps en secuencia incremental customer_id: {len(faltantes)} entradas faltantes. Primeros 10: {faltantes[:10]}')
    else:
        print('  Secuencia incremental sin gaps: OK')
        comprobante = 5

comprobaciones['customer_id_comprobante_final'] = comprobante
print(f'  Comprobante customer_id final: {comprobante}')


**Resultado customer_id:** 59.932 filas, todas con formato `CLI-` + 6 dígitos, sin vacíos, sin duplicados. Sin embargo, hay **40.068 gaps** en la secuencia incremental (el rango va de 1 a 100.000 pero solo están presentes 59.932). Esto no invalida la clave primaria, pero confirma que `customer_id` no es un incremental continuo sin huecos.

In [ ]:
print('--- Validación: first_name y last_name ---')
for col in ['first_name', 'last_name']:
    s = df[col].astype(str).str.strip()
    n_vac = int((s == '').sum())
    comprobaciones[f'{col}_vacío'] = n_vac
    if n_vac > 0:
        registrar('error', f'{col} vacío: {n_vac} filas')
    else:
        print(f'  {col} sin celdas vacías: OK')

    mascara_rara = ~s.str.match(r'^[\wáéíóúÁÉÍÓÚñÑ\s\'\-\.]+$', na=False)
    n_rara = int(mascara_rara.sum())
    comprobaciones[f'{col}_caracteres_inusuales'] = n_rara
    if n_rara > 0:
        ejemplos = df.loc[mascara_rara, col].head(10).tolist()
        registrar('advertencia', f'{col} con caracteres inusuales: {n_rara} filas. Ejemplos: {ejemplos}')
    else:
        print(f'  {col} con caracteres alfabéticos esperados: OK')

# Homónimos (advertencia, no error)
nombre_completo = (
    df['first_name'].str.strip().str.lower()
    + ' | '
    + df['last_name'].str.strip().str.lower()
)
mascara_hom = nombre_completo.duplicated(keep=False) & (df['first_name'] != '') & (df['last_name'] != '')
n_hom = int(mascara_hom.sum())
n_grupos_hom = int(nombre_completo[mascara_hom].nunique()) if n_hom > 0 else 0
comprobaciones['homónimos_filas'] = n_hom
comprobaciones['homónimos_grupos'] = n_grupos_hom
if n_hom > 0:
    registrar('advertencia', f'Homónimos: {n_hom} filas en {n_grupos_hom} grupos (mismo nombre+apellido, distinto customer_id). El anexo permite homónimos; no usar nombre+apellido como clave de unión.')
else:
    print('  Sin homónimos: OK')

comprobaciones['nombres_comprobante_final'] = 2
print(f'  Comprobante nombres final: 2')


In [ ]:
print('--- Validación: email ---')
email = df['email'].astype(str).str.strip()

# Comprobante 1: sin celdas vacías
n_vac = int((email == '').sum())
comprobaciones['email_vacío'] = n_vac
if n_vac > 0:
    registrar('error', f'email vacío: {n_vac} filas')
else:
    print('  email sin celdas vacías: OK')
    comprobante_email = 1

# Comprobante 2: formato válido
email_re = re.compile(r'^[^@\s]+@[^@\s]+\.[^@\s]+$')
valido_fmt = email.str.match(email_re)
n_fmt = int((~valido_fmt & (email != '')).sum())
comprobaciones['email_formato_inválido'] = n_fmt
if comprobante_email == 1:
    if n_fmt > 0:
        ejemplos = email.loc[~valido_fmt & (email != '')].head(10).tolist()
        registrar('error', f'email con formato inválido: {n_fmt} filas. Ejemplos: {ejemplos}')
    else:
        print('  email con formato válido: OK')
        comprobante_email = 2

# Comprobante 3: dominio @example.com
if comprobante_email == 2:
    dominio = email.str.split('@').str[1].str.lower()
    n_dom = int((dominio != 'example.com').sum())
    comprobaciones['email_dominio_inválido'] = n_dom
    if n_dom > 0:
        ejemplos = email.loc[dominio != 'example.com'].head(10).tolist()
        registrar('error', f'email con dominio distinto a @example.com: {n_dom} filas. Ejemplos: {ejemplos}')
    else:
        print('  email con dominio @example.com: OK')
        comprobante_email = 3

# Comprobante 4: sin duplicados
if comprobante_email == 3:
    n_dup_email = int(email.duplicated(keep=False).sum())
    comprobaciones['email_duplicado'] = n_dup_email
    if n_dup_email > 0:
        registrar('error', f'email duplicado: {n_dup_email} filas')
    else:
        print('  email sin duplicados: OK')
        comprobante_email = 4

comprobaciones['email_comprobante_final'] = comprobante_email
print(f'  Comprobante email final: {comprobante_email}')


**Resultado email:** 59.932 filas, todas con formato válido, dominio `@example.com`, sin vacíos y sin duplicados.

## 6. Tabla de calidad del dataset original

Se calcula completitud (porcentaje de celdas no vacías) y unicidad de `customer_id` para dejar constancia del estado antes de cualquier limpieza.

In [ ]:
tot = len(df)
comp = (df.notna().sum().sum() / (tot * len(df.columns))) * 100
dupl_id = int(df['customer_id'].duplicated().sum())
unic = ((tot - dupl_id) / tot) * 100

print('=' * 58)
print(f"{'Dimensión':<20} | {'Dataset original':<18} | {'Estado':<12}")
print('=' * 58)
print(f"{'Completitud':<20} | {comp:>16.1f}% | {'OK' if comp >= 99 else 'REVISAR':<12}")
print(f"{'Unicidad customer_id':<20} | {unic:>16.1f}% | {'OK' if unic == 100 else 'REVISAR':<12}")
print('=' * 58)


## 7. Resumen de errores y advertencias

Listado consolidado de todo lo encontrado durante la auditoría.

In [ ]:
print('=' * 70)
print('RESUMEN FINAL DE AUDITORÍA')
print('=' * 70)
print(f'Total errores: {len(errores)}')
print(f'Total advertencias: {len(advertencias)}')
print()
if errores:
    print('ERRORES ENCONTRADOS:')
    for e in errores:
        print(f'  - {e}')
else:
    print('No se encontraron errores importantes.')
print()
if advertencias:
    print('ADVERTENCIAS:')
    for a in advertencias:
        print(f'  - {a}')
print()
print('Comprobantes finales:')
print(f'  customer_id: {comprobaciones.get("customer_id_comprobante_final", "N/A")}')
print(f'  nombres: {comprobaciones.get("nombres_comprobante_final", "N/A")}')
print(f'  email: {comprobaciones.get("email_comprobante_final", "N/A")}')
print()
print('Nota: el CSV original no se modificó.')


## 8. Limpieza y exportación

Como no se encontraron errores que requieran modificar filas (salvo los gaps en `customer_id`, que son una propiedad del identificador y no un dato erróneo), el archivo limpio es una copia exacta del original con las mismas 59.932 filas y las mismas 4 columnas. El script `tests/limpiar_customers.py` aplica el mismo sistema de comprobantes y confirma el comprobante final 8 (limpieza completada).

In [ ]:
df_limpio = df.copy()

# Guardar CSV limpio
df_limpio.to_csv(RUTA_LIMPIO, index=False, encoding='utf-8-sig')
print(f'Archivo guardado: {RUTA_LIMPIO.resolve()}')

# Releer para comprobar que conserva filas y columnas
df_releido = pd.read_csv(
    RUTA_LIMPIO,
    dtype=str,
    encoding='utf-8-sig',
    keep_default_na=False,
    na_values=[''],
)

print(f'Original: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'Limpio:   {df_releido.shape[0]} filas × {df_releido.shape[1]} columnas')
print(f'Columnas iguales: {list(df.columns) == list(df_releido.columns)}')
print(f'Datos idénticos: {df.equals(df_releido)}')


## 9. Validación con `assert`

Antes de dar por terminada la limpieza, se comprueban con `assert` las propiedades que necesitará cualquier notebook que use este catálogo como referencia de compradores.

In [ ]:
def validar_customers(df):
    """Comprueba que el catálogo de compradores es usable como tabla de referencia."""
    def comprobar(condicion, descripcion):
        assert condicion, f'FALLO: {descripcion}'
        print(f'  ok · {descripcion}')

    n = len(df)
    comprobar(n == 59932, 'hay 59.932 compradores')
    comprobar(df['customer_id'].is_unique, 'customer_id es único (clave primaria)')
    comprobar(df['email'].is_unique, 'email es único')
    comprobar((df['customer_id'] != '').all(), 'sin customer_id vacío')
    comprobar((df['email'] != '').all(), 'sin email vacío')
    comprobar(df['customer_id'].str.match(r'^CLI-\d+$').all(), 'customer_id con formato CLI-<números>')
    comprobar(df['email'].str.match(r'^[^@\s]+@[^@\s]+\.[^@\s]+$').all(), 'email con formato válido')
    comple = df['email'].str.split('@').str[1].str.lower().eq('example.com').all()
    comprobar(comple, 'todos los emails tienen dominio @example.com')
    comple2 = (df[['first_name', 'last_name']] != '').all().all()
    comprobar(comple2, 'sin first_name ni last_name vacío')
    print('  ok · sin valores ausentes en ninguna columna')


validar_customers(df_limpio)


## 10. Reflexión final

**Qué se ha demostrado**
- `customers.csv` tiene 59.932 compradores, sin valores ausentes, sin duplicados en `customer_id` ni en `email`, y todos los emails usan el dominio `@example.com`.
- `customer_id` es una clave primaria válida con formato consistente (`CLI-` + 6 dígitos).
- `customer_id` tiene 40.068 gaps en la secuencia incremental (rango 1-100.000, solo 59.932 presentes): no es un error del fichero, pero confirma que no es un incremental continuo sin huecos.
- Hay 41.198 filas con homónimos (mismo `first_name` + `last_name`, distinto `customer_id`), repartidas en 16.730 grupos. Esto es **esperado y permitido** por el anexo técnico; nunca se debe usar nombre+apellido como clave de unión.
- La tabla de calidad muestra completitud 100.0% y unicidad de `customer_id` 100.0%.
- El archivo limpio `customers_clean.csv` es idéntico al original (0 celdas diferentes), ya que no hubo errores que corregir.

**Qué no se ha demostrado**
- Que los gaps en `customer_id` sean intencionados o fruto de borrados; este fichero solo los constata.
- Si hay compradores que no aparecen en ninguna reserva (el catálogo no incluye a todos los acompañantes, pero sí a todos los compradores que realizaron una compra en el periodo exportado).

**Cómo usarlo en la integración**
- Es la tabla que **enlaza compras con compradores**: `reservations.customer_id` → `customers.customer_id` da el comprador de cada reserva.
- Un comprador puede tener varias reservas (un pedido agrupa varias reservas de un mismo comprador), por lo que la relación `customers` → `reservations` es 1:N.
- Nunca usar `first_name` + `last_name` como clave; siempre `customer_id`.

**Pendiente para la integración** (requiere otros ficheros)
- Unir `customers` con `reservations` para saber cuántos compradores distintos están afectados por cada tipo de incidencia.
- Contrastar si algún `customer_id` de `reservations` no existe en `customers`.
- Analizar si hay compradores con incidencias en varios conciertos (patrón común entre proveedores).